# Wst?p do Uczenia Maszynowego - Projekt 1
## Autorzy: Katarzyna Solawa, Jan Smole?


Tematem naszego projektu jest przewidywanie przynale?no?ci partyjnej cz?onka Izby Reprezentant?w ameryka?skiego kongresu w 1986 roku na podstawie dokonanych przez niego wybor?w podczas g?osowa?. Naszym zbiorem danych jest ramka zawieraj?ca dane o przynale?no?ci partyjnej poszczeg?lnych reprezentant?w i ich g?osach podczas 16 kluczowych w tym roku g?osowa?. 

# Importy

In [ ]:
import pandas as pd 
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import sklearn.metrics
import random
from sklearn import manifold
random.seed(42)
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVC
import xgboost as xgb
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import plot_precision_recall_curve
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
import sklearn.metrics as metrics
import statistics

# EDA

In [ ]:
df=pd.read_csv("congressional_voting_dataset.csv")

In [ ]:
df.info()

In [ ]:
df.head()

## Obja?nienie zmiennych
Kolumny 0-15 zawieraj? wyniki g?osowa? na tematy skr?towo opisane w nazwach kolumn. Ka?dy rz?d odpowiada jednemu reprezentantowi. Mo?liwe warto?ci: <br>
**y** - g?os na tak <br>
**n** - g?os na nie <br>
**?** - brak g?osu - niewzi?cie udzia?u w g?osowaniu lub wstrzymanie si? od g?osu <br>
Ostatnia kolumna zawiera informacje o przynale?no?ci partyjnej reprezentanta - **republican** albo **democrat**. W naszej ramce danych nie wyst?puje bezpo?rednio problem braku danych, ale zapewne b?dzie trzeba jako? rozwi?za? kwesti? warto?ci **?**.

In [ ]:
df.describe()

In [ ]:
labels=["y", "n", "?"]
fig, axs = plt.subplots(ncols=2, nrows=8, figsize=(16, 32))
for i in range(len(df.columns)-1):
    col=df.columns[i]
    tmp=df[[col, "political_party"]].groupby(["political_party", col]).size().tolist()
    r, c= i//2, i%2
    axs[r,c].bar(labels, list(reversed(tmp[0:3])), label='democrat', color="blue")
    axs[r,c].bar(labels, list(reversed(tmp[3:6])), bottom=list(reversed(tmp[0:3])),
       label='republican', color="red")
    axs[r,c].legend()
    axs[r,c].set_title(col)

Obie partie g?osowa?y podobnie na `water_project_cost_sharing` oraz `imigration`(lecz u demokrat?w przewa?a `no`, a u republikan `yes`)
Widoczna r?znica g?os?w dla:
- `adoption_of_the_budget_resolution`(r-no, d-yes)
- `physician_fee_freeze`(r-yes, d-no)
- `el_salvador_aid`(r-yes, d-no)
- `education_spending`(r-yes, d-no)

In [ ]:
df=df.replace("n", 0)
df=df.replace("y", 1)
df=df.replace("?",  0.5)   #rozwi?zanie tymczasowe
df=df.replace("republican", 0)
df=df.replace("democrat",  1)
plt.figure(figsize=(10,10))
sns.heatmap(df.corr(), annot=True, annot_kws={'size': 8}, fmt='.2f')
plt.show()

Jak widzimy, poziom korelacjii pomi?dzy g?osem a parti? bardzo si? r??ni w zale?no?ci od tematu g?osowania - dla g?osowania **water_project_cost_sharing** zwi?zek praktycznie nie istnieje, a dla **physician_fee_freeze** jest bardzo du?y.

Spr?bujemy teraz zobaczy?, na ile g?osy poszczeg?lnych reprezentant?w przypominaj? g?osy innych cz?onk?w tej samej partii - w tym celu przekszta?cimy zapisy g?osowa? poszczeg?lnych cz?onk?w na wektory i policzymy odleg?o?ci pomi?dzy ka?d? par?. 

In [ ]:
adist=sklearn.metrics.pairwise_distances(df.drop(["political_party"], axis=1))
adist

U?yjemy teraz funkcji z pakietu manifold ?eby przekszta?ci? ramk? zawieraj?c? wzajemne odleg?o?ci na zbi?r wsp??rz?dnych na dwuwymiarowej p?aszczy?nie. Jest to rzut, kt?ry pr?buje przekszta?ci? wielowymiarowe zale?no?ci na p?aszczyzn? 2D. 

In [ ]:
df["political_party"]=df["political_party"].replace(0, "republican")
df["political_party"]=df["political_party"].replace(1, "democrat")
adist=np.array(adist)
mds = manifold.MDS(n_components=2, dissimilarity="precomputed", random_state=6)
results = mds.fit(adist)
coords = results.embedding_
fig, ax = plt.subplots(figsize=(10,10))
sns.scatterplot(
    coords[:, 0], coords[:, 1], marker = 'o', hue=df["political_party"], palette=["red", "blue"]
    )
ax.set_title("Voting pattern similarity")

Dodatkowo sprawd?my czy kt?ra? z parti ma sko?onno?? do g?osowania na tak lub nie.

In [ ]:
df=pd.read_csv("congressional_voting_dataset.csv")
democrat_df = df[df['political_party'] == 'democrat']
republican_df = df[df['political_party'] == 'republican']

In [ ]:
tak = 0
nie = 0
brak = 0
for i in range(0,15):
    tak += (democrat_df[democrat_df.columns[i]] == "y").sum()
    nie += (democrat_df[democrat_df.columns[i]] == "n").sum()
    brak += (democrat_df[democrat_df.columns[i]] == "?").sum()

labels = ['Yes', 'No', '?']
sizes = [tak, nie,brak]
plt.title("Democrat")
plt.bar(labels, sizes)
plt.show()

In [ ]:
tak = 0
nie = 0
brak = 0
for i in range(0,15):
    tak += (republican_df[republican_df.columns[i]] == "y").sum()
    nie += (republican_df[republican_df.columns[i]] == "n").sum()
    brak += (republican_df[republican_df.columns[i]] == "?").sum()

labels = ['Yes', 'No', '?']
sizes = [tak, nie,brak]
plt.title("Republican")
plt.bar(labels, sizes)
plt.show()

W obu przypadkach licza g?os?w jest do?? wyr?wnana.

# Feature Engineering

In [ ]:
import pandas as pd 
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import sklearn.metrics
import random
from sklearn import manifold
import xgboost as xgb
random.seed(42)
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import chi2
from matplotlib import pyplot 

In [ ]:
df=pd.read_csv("congressional_voting_dataset.csv")

### Encoding
W naszych danych kodowanie zmiennych kategorycznych wydaje si? nie by? du?ym wyzwaniem. G?osy na nie oznaczamy jako 0, brak g?osu  jako 0.5, a g?osy na tak to 1. Podobnie intuicyjnie republikan?w oznaczamy jako zera, a demokrat?w jako jedynki.

In [ ]:
df=df.replace("n", 0)
df=df.replace("y", 1)
df=df.replace("?",  0.5)  
df=df.replace("republican", 0)
df=df.replace("democrat",  1)


### Outliers
Ze wzgl?du na kategoryczne warto?ci w naszych danych, nie widzimy tu outlier?w w postaci rz?d?w, kt?re si? szczeg?lnie wyr??niaj? jedn? warto?ci?. Jedyny rz?d, kt?ry odrzucimy to ten, w kt?rym warto?ci wszystkich g?osowa? wynosi?y "?" - jest to prawdopodobnie brak danych, b?d? dany reprezentant z jaki? osobliwych powod?w nie wzi?? udzia?u w ?adnym g?osowaniu. 

In [ ]:
X=df.drop(["political_party"], axis=1)
indexes=[]
colnames=X.columns
for i in range(len(X)):
    for j in range(len(colnames)):
        if X.iloc[i, j]!=0.5:
            break
        if j==len(colnames)-1:
            indexes.append(i)
X=X.drop(248, axis=0)
y=df["political_party"].drop(248, axis=0)
df=df.drop(248, axis=0)
df.to_csv("df_encoded.csv", index=False)

# Feature Selection
Na pocz?tku korzystali?my z bardziej intycuicyjnych sposob?w wybor?w cech, a potem zastosowali?my metody pokazane na laboratoriach. Spojrzymy najpierw ponownie na macierz korelacji.


In [ ]:
plt.figure(figsize=(10,10))
sns.heatmap(df.corr(), annot=True, annot_kws={'size': 8}, fmt='.2f')
plt.show()

Usuniemy dwie zmienne, kt?re w por?wnaniu z innymi s? bardzo ma?o skorelowane z naszym celem - **water_project_cost_sharing** i **immigration**. Spr?bujemy te? usun?? zmienn? **el_salvador_aid** - mimo, ?e jest silnie zwi?zana z celem, jest tak?e najbardziej skorelowana z innymi zmiennymi obja?niaj?cymi.

In [ ]:
drop=["water_project_cost_sharing", "immigration", "el_salvador_aid"]
X=X.drop(drop, axis=1)

### Model
Naszym modelem bazowym, kt?rego dzi? u?yjemy, b?dzie xgboost bez tuningu hiperparametr?w.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size = 0.2, random_state = 42)

In [ ]:
xgb_model = xgb.XGBClassifier(objective = "binary:logistic", seed = 42, use_label_encoder=False, verbosity=0)
xgb_model.fit(X_train, y_train)
preds = xgb_model.predict(X_test)
comparison = pd.DataFrame({'actual':y_test, 'predicted':preds})
print("Accuracy: " + str(sum(comparison["actual"] == comparison["predicted"]) / len(comparison) * 100) + "%")

Jak widzimy, nasz model ca?kiem dobrze sobie radzi z przewidywaniem przynale?no?ci do danej partii politycznej - osi?ga ponad 96% skuteczno?ci. Na koniec spojrzymy, jak wa?ne dla niego s? poszczeg?lne kolumny - u?yjemy do tego wbudowanej funkcji modelu xgb. 

In [ ]:
plt.figure(figsize=(8,8))
pyplot.bar(X.columns, xgb_model.feature_importances_)
plt.xticks(rotation=90)
pyplot.show()

Zgodnie z oczekiwaniami wynikaj?cymi z mapy korelacji, zmienna **physicican_fee_freeze** ma olbrzymi wp?yw na predykcje naszego modelu. Na koniec sp?jrzmy jeszcze, jak wygl?da?a by skuteczno?? modelu, gdyby?y wybrali jedynie 5 najwa?niejszych wg wykresu cech.

In [ ]:
fts=["physician_fee_freeze", "mx_missile", "synfuels_corporation_cutback", "religious_groups_in_schools", "crime"]
X=X[fts]
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size = 0.2, random_state = 42)
xgb_model = xgb.XGBClassifier(objective = "binary:logistic", seed = 42, use_label_encoder=False, verbosity=0)
xgb_model.fit(X_train, y_train)
preds = xgb_model.predict(X_test)
comparison = pd.DataFrame({'actual':y_test, 'predicted':preds})
print("Accuracy: " + str(sum(comparison["actual"] == comparison["predicted"]) / len(comparison) * 100) + "%")


In [ ]:
import pandas as pd 
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import sklearn.metrics
import random
from sklearn import manifold
random.seed(42)


In [ ]:
df=pd.read_csv("congressional_voting_dataset.csv")
df=df.replace("n", 0)
df=df.replace("y", 1)
df=df.replace("?",  0.5)  
df=df.replace("republican", 0)
df=df.replace("democrat",  1)
X=df.drop(["political_party"], axis=1)
indexes=[]
colnames=X.columns
for i in range(len(X)):
    for j in range(len(colnames)):
        if X.iloc[i, j]!=0.5:
            break
        if j==len(colnames)-1:
            indexes.append(i)
X=X.drop(248, axis=0)
y=df["political_party"].drop(248, axis=0)
df.head()

In [ ]:
drop=["water_project_cost_sharing", "immigration", "el_salvador_aid"]
X=X.drop(drop, axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size = 0.2, random_state = 42)
X_best=None

In [ ]:
def selectFeature(selector):
    selector.fit(X_train, y_train)
    X_train_fs = selector.transform(X_train)
    X_test_fs = selector.transform(X_test)
    xgb_model = xgb.XGBClassifier(objective = "binary:logistic", seed = 42, use_label_encoder=False, verbosity=0)
    xgb_model.fit(X_train_fs, y_train)
    yhat = xgb_model.predict(X_test_fs)
    accuracy = accuracy_score(y_test, yhat)
    print('Accuracy: %.2f' % (accuracy*100))


In [ ]:
def selectFeature2(selector):
    clf = Pipeline([
      ('feature_selection', SelectFromModel(selector)),
      ('classification', xgb.XGBClassifier(objective = "binary:logistic", seed = 42, use_label_encoder=False, verbosity=0))
    ])
    clf.fit(X_train, y_train)
    yhat = clf.predict(X_test)
    accuracy = accuracy_score(y_test, yhat)
    print('Accuracy: %.2f' % (accuracy*100))

### CHI2

In [ ]:
for i in range(1,14):
    selectFeature(SelectKBest(chi2, k=i))    


Najwy?sze accuracy dla wyboru wszytkich 13 zmiennych

### Recursive Feature Elimination

In [ ]:
from sklearn.feature_selection import RFE
estimator = LogisticRegression()
for i in range(1,14):
    selectFeature(RFE(estimator, n_features_to_select=i, step=1))

 

Najwy?sze accuracy przy wyborze min 4 zmiennych

### Mutual info classif

In [ ]:
for i in range(1,14):
    selectFeature(SelectKBest(mutual_info_classif, k=i))

Najwy?sze accuracy przy wyborze min 10 zmiennych

### LinearSVC - Linear Support Vector Classification

In [ ]:

# C = 0.005
selectFeature2(LinearSVC(C=0.005, penalty="l1", dual=False))

# C = 0.005
selectFeature2(LinearSVC(C=0.01, penalty="l1", dual=False))

# C = 0.11 
selectFeature2(LinearSVC(C=0.11, penalty="l1", dual=False))

# C = 0.17 
selectFeature2(LinearSVC(C=0.17, penalty="l1", dual=False))

# C = 0.2
selectFeature2(LinearSVC(C=0.2, penalty="l1", dual=False))



In [ ]:
selectFeature2(LinearSVC())#domy?lnie C = 1, penalty='l2'

### PolynomialFeatures

In [ ]:
pf = PolynomialFeatures(degree=3)
pf.fit(X_train, y_train)
X_train_fs = pf.transform(X_train)
X_test_fs = pf.transform(X_test)
xgb_model = xgb.XGBClassifier(objective = "binary:logistic", seed = 42, use_label_encoder=False, verbosity=0)
xgb_model.fit(X_train_fs, y_train)
yhat = xgb_model.predict(X_test_fs)
accuracy = accuracy_score(y_test, yhat)
print('Accuracy: %.2f' % (accuracy*100))

In [ ]:
def selectFeaturePF(selector):
    pf = PolynomialFeatures(degree=3)
    pf.fit(X_train, y_train)
    X_train_pf = pf.transform(X_train)
    X_test_pf = pf.transform(X_test)
    selector.fit(X_train_pf, y_train)
    X_train_fs = selector.transform(X_train_pf)
    X_test_fs = selector.transform(X_test_pf)
    xgb_model = xgb.XGBClassifier(objective = "binary:logistic", seed = 42, use_label_encoder=False, verbosity=0)
    xgb_model.fit(X_train_fs, y_train)
    yhat = xgb_model.predict(X_test_fs)
    accuracy = accuracy_score(y_test, yhat)
    print('Accuracy: %.2f' % (accuracy*100))

In [ ]:
for i in range(1,20):
    print(f'i = {i}')
    selectFeaturePF(SelectKBest(chi2, k=i))   
    
print(f'i = {42}')
selectFeaturePF(SelectKBest(chi2, k=42))

In [ ]:
for i in range(1,20):
    print(f'i = {i}')
    selectFeaturePF(SelectKBest(mutual_info_classif, k=i))
    
print(f'i = {37}')
selectFeaturePF(SelectKBest(mutual_info_classif, k=37))

Przy wyborze do k najlepszych zmiennych za pomoc? chi2 oraz mutual info classif, accuracy jest wi?ksze ni? 94.25 dla k min r?wnego 42 i 37.

## Wnioski

Metody osi?gaj? max accuracy 96.55, czyli tyle ile uda?o nam sie osi?gn?? przy samodzielnym wyborze zmiennych. </br>  
U?ywaj?c niedu?ej ilo?ci zmiennych uda?o nam si? zatem osi?gn?? wynik tylko nieznacznie gorszy od bazowego XGBoosta, kt?ry wynosi? 97%. Jednak ze wzgl?du na ma?? liczb? rekord?w i kr?tki czas wykonywania algorytm?w, nie widzeli?my sensu w ograniczaniu w tym przypadku liczby kolumn.

In [ ]:
fts=["physician_fee_freeze","synfuels_corporation_cutback", "adoption_of_the_budget_resolution", "education_spending"]
X2=X[fts]
X_train, X_test, y_train, y_test = train_test_split(X2, y, stratify=y, test_size = 0.2, random_state = 42)
xgb_model = xgb.XGBClassifier(objective = "binary:logistic", seed = 42, use_label_encoder=False, verbosity=0)
xgb_model.fit(X_train, y_train)
preds = xgb_model.predict(X_test)
comparison = pd.DataFrame({'actual':y_test, 'predicted':preds})
print("Accuracy: " + str(sum(comparison["actual"] == comparison["predicted"]) / len(comparison) * 100) + "%")

In [ ]:
xgb_tmp=xgb.XGBClassifier(objective = "binary:logistic", seed = 42, use_label_encoder=False, verbosity=0)

In [ ]:
xgb_tmp

In [ ]:
y_train

# Wyb?r modelu 

In [ ]:
df=pd.read_csv("df_encoded.csv")

Chocia? na we wcze?niejszym etapie pracy usun?li?my z naszych danych zduplikowane rz?dy, to podczas pracy z modelami okazuje si?, ?e osi?gaj? one lepsze wyniki kiedy je zostawimy.

In [ ]:
from sklearn.model_selection import train_test_split
X=df.drop(["political_party"], axis=1)
y=df["political_party"]
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y,random_state = 42)

Wybrali?my do przetestowania 3 modele - **SVM**, **Random Forest** i **XGBoost**.

## SVM

In [ ]:
svm_base=SVC(random_state=42)
svm_base.fit(X_train, y_train)
preds=svm_base.predict(X_test)

In [ ]:
svm_base_acc=accuracy_score(preds,y_test)
print("Accuracy SVM z domy?lnymi hiperparametrami: " + str(svm_base_acc))

Widzimy, ?e ju? domy?lny SVM osi?ga bardzo dobre accuracy na poziomie ponad 96%. Spr?bujemy teraz wykona? tuning hiperparametr?w. Poniewa? nasz zbi?r danych jest stosunkowo ma?y, skorzystamy z narz?dzia GridSearch.

In [ ]:

svm_tuned=SVC(random_state=42)
c=[]  # warto?ci parametru C
gamma=[]  #warto?ci parametru gamma 
for i in range(-4, 5):      # orientacyjne warto?ci na podstawie informacji znalezionych w internecie
    c.append(10**i)
for i in range(-4, 5):
    gamma.append(10**i)
gamma.append("auto")
gamma.append("scale")
params = [{'C': c,   
           "kernel": ["rbf", "linear", "poly"],
        'gamma': gamma}]
gs_svm=GridSearchCV(svm_tuned, param_grid=params, scoring='accuracy', cv=4, n_jobs=2)
gs_svm.fit(X_train, y_train)
gs_svm.best_params_

In [ ]:
svm_tuned_acc=accuracy_score(gs_svm.predict(X_test),y_test)
print("Accuracy SVM po tuningu hiperparametr?w: " + str(svm_tuned_acc))

Jak widzimy, nie uda?o nam si? polepszy? wyniku, a nawet uzyskali?my accuracy troch? gorsze. Mimo ?e w naszej ramce danych znajduj? si? tak?e domy?lne warto?ci hiperparametr?w, to wypad?y one gorzej przy kroswalidacji i dlatego algorytm ich nie wybra?. Wydaje mi si?, ?e taka sytuacja zachodzi ze wzgl?du na ma?? liczb? rekord?w i du?e accuracy naszych modeli.

## XGBoost

In [ ]:
xgb_base = xgb.XGBClassifier(objective = "binary:logistic", seed = 1613, use_label_encoder=False, verbosity=0)
xgb_base.fit(X_train, y_train)
preds=xgb_base.predict(X_test)

In [ ]:
xgb_base_acc=accuracy_score(preds,y_test)
print("Accuracy XGB z domy?lnymi hiperparametrami: " + str(xgb_base_acc))

In [ ]:
xgb_tuned=xgb.XGBClassifier(objective = "binary:logistic", seed = 1613, use_label_encoder=False, eval_metric="error")
params = {
        'min_child_weight': [1, 5, 10],
        'gamma': [0.5, 1, 1.5, 2, 5],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'max_depth': [3, 4, 5]
        }

gs_xgb=GridSearchCV(xgb_tuned, param_grid=params, scoring='accuracy', cv=4, n_jobs=2)
gs_xgb.fit(X_train, y_train)
gs_xgb.best_params_

In [ ]:
xgb_tuned_acc=accuracy_score(gs_xgb.predict(X_test), y_test)
print("Accuracy XGB po treningu hiperparametr?w: " + str(xgb_tuned_acc))

## Random Forest

In [ ]:
rfc_base = RandomForestClassifier(random_state=16)
rfc_base.fit(X_train, y_train)
preds=rfc_base.predict(X_test)

In [ ]:
rfc_base_acc=accuracy_score(preds,y_test)
print("Accuracy RFC z domy?lnymi hiperparametrami: " + str(rfc_base_acc))

In [ ]:
rfc_tuned=RandomForestClassifier(random_state=16)
n_estimators = [int(x) for x in np.linspace(start = 50, stop = 1000, num = 5)] # przyk?adowe warto?ci znalezione w internecie 
max_depth = [int(x) for x in np.linspace(5, 55, num = 5)]
max_features= ['auto', 'sqrt', 'log2']
             
params = [{'n_estimators': n_estimators,
        'max_depth': max_depth,
          'max_features': max_features}]
gs_rfc=GridSearchCV(rfc_tuned, param_grid=params, scoring='accuracy', cv=4, n_jobs=2)
gs_rfc.fit(X_train, y_train)
gs_rfc.best_params_

In [ ]:
rfc_tuned_acc=accuracy_score(gs_rfc.predict(X_test), y_test)
print("Accuracy RFC po treningu hiperparametr?w: " + str(rfc_tuned_acc))

Jak wida?, ani w XGBoo?cie, ani w Random Forest nie uda?o si? uzyska? lepszej ni? domy?lna accuracy. Co wi?cej, obydwa bazowe modele osi?gaj? dok?adnie ten sam wynik.

# Ocena modeli
## Accuracy score

In [ ]:
scores=[]
labels=[]
scores.append(svm_base_acc)
labels.append("SVM")
scores.append(xgb_base_acc)
labels.append("XGB")
scores.append(rfc_base_acc)
labels.append("RFC")

In [ ]:
pd.DataFrame({"Accuracy Score": scores}, index=labels)

## Confusion matrix
### SVM

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, svm_base.predict(X_test)).ravel()
pd.DataFrame({"Actual positives": [tp, fp], "Actual negatives": [fn, tn]}, index = ["Positive predictions", "Negative predictions"])

### XGB 

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, xgb_base.predict(X_test)).ravel()
pd.DataFrame({"Actual positives": [tp, fp], "Actual negatives": [fn, tn]}, index = ["Positive predictions", "Negative predictions"])

### RFC

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, xgb_base.predict(X_test)).ravel()
pd.DataFrame({"Actual positives": [tp, fp], "Actual negatives": [fn, tn]}, index = ["Positive predictions", "Negative predictions"])

## ROC AUC

In [ ]:
gs_svm
plt.figure(figsize=(12,10))
classifiers = [svm_base, xgb_base, rfc_base]
labels=["SVM", "XGB", "RFC"]
ax = plt.gca()
for i in range(3):
    metrics.plot_roc_curve(classifiers[i], X_test, y_test, ax=ax, name=labels[i])

Na podstawie powy?szych wynik?w zdecydowali?my si? pozosta? przy modelu **XGBoost** bez modyfikacji hiperparametr?w czy wyboru zmiennych. W naszym przypadku nawet taki model osiaga? bardzo wysokie accuracy, a by? przy tym szybki.

# Podsumowanie 
## EDA throwback - kt?re predykcje si? uda?y


In [ ]:
df_tmp=pd.read_csv("congressional_voting_dataset.csv")
df_tmp=df_tmp.replace("n", 0)
df_tmp=df_tmp.replace("y", 1)
df_tmp=df_tmp.replace("?",  0.5)                                                  #?eby uzyska? taki sam wykres jak w eda
df_tmp["political_party"]=df_tmp["political_party"].replace("republican", 0)      #musimy u?y? ramki danych z rz?dem,
df_tmp["political_party"]=df_tmp["political_party"].replace("democrat", 1)        # kt?ry wcze?niej wyrzucli?my

y_tmp=df_tmp["political_party"]
X_tmp=df_tmp.drop(["political_party"], axis=1)


xgb_tmp=xgb.XGBClassifier(objective = "binary:logistic", seed = 1613, use_label_encoder=False, verbosity=0)
df_tmp["predicted_correctly"]=False
for i in range(len(df_tmp)):
    X_train_tmp=X_tmp.drop(i, axis=0)
    y_train_tmp=y_tmp.drop(i, axis=0)
    X_test_tmp=X_tmp.iloc[[i]]
    y_test_tmp=y_tmp[i]
    xgb_tmp.fit(X_train_tmp, y_train_tmp)
    pred_tmp=xgb_tmp.predict(X_test_tmp)[0]
    if y_tmp[i]==pred_tmp:
        df_tmp["predicted_correctly"][i]=True
    

In [ ]:
adist=sklearn.metrics.pairwise_distances(df_tmp.drop(["political_party", "predicted_correctly"], axis=1))
adist

In [ ]:
df_tmp["political_party"]=df_tmp["political_party"].replace(0, "republican")
df_tmp["political_party"]=df_tmp["political_party"].replace(1, "democrat")
adist=np.array(adist)
mds = manifold.MDS(n_components=2, dissimilarity="precomputed", random_state=6)
results = mds.fit(adist)
coords = results.embedding_
fig, ax = plt.subplots(figsize=(12,12))
sns.scatterplot(
    coords[:, 0], coords[:, 1], marker = 'o', hue=df_tmp["political_party"], palette=["red", "blue"], 
    style=df_tmp["predicted_correctly"], markers=["X", "o"], size=df_tmp["predicted_correctly"]
    )
ax.set_title("Voting pattern similarity")

Na koniec wr?cili?my do wykresu z naszego EDA ?eby zobaczy?, kt?rych z reprezentant?w uda?o przewidzie? si? prawid?owo ? wykonali?my w tym celu predykcje na podstawie reszty ramki danych dla ka?dego pojedynczego rz?du. Jak wida?, b??dne predykcje zazwyczaj znajduj? si? w zr??nicowanym pod wzgl?dem partii otoczeniu. Jednak du?a skuteczno?? nawet dla nieoczywistych polityk?w  pokazuje ?wiadczy o jako?ci XGBoosta, nawet bez specjalnego dopasowywania cech czy tuningu hiperparametr?w. 